# 02 Transcript context: the segmentation-free molecular field

Segmentation assigns each transcript to one cell. **Transcript context** instead counts every molecule within a small radius of a cell's centroid, regardless of segmentation, giving a local molecular field that is robust to segmentation errors and captures the immediate microenvironment. We concatenate it onto the segmented counts, doubling the input dimension, and train the same hierarchical VQ-VAE.

Here we use the bundled **real Xenium RCC TMA core** (7,824 cells, 366-gene panel), which ships with its matched molecule table.

**Demo note:** ~20 epochs for speed; a real cohort uses `num_epochs ~300`. The default radius is **7 um** (measured to capture roughly one cell from its nucleus center with minimal neighbor bleed).

In [ ]:
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
import nicheverse as nv
from nicheverse import ModelConfig, TrainConfig
from nicheverse.data import transcript_context
import os
DATA = os.path.join('..', 'examples', 'data')
adata = nv.read_spatial(f'{DATA}/xenium_rcc_core.h5ad', sample_col='sample_id')
print(adata.n_obs, 'cells x', adata.n_vars, 'genes')

## Compute the transcript-context field

`transcript_context` reads a per-sample molecule table (`x_location`, `y_location`, `feature_name` for Xenium), drops control / blank probes, and for each cell counts the molecules within `radius` microns of its centroid, returning a log1p `(n_cells, n_genes)` matrix. Pass a single path for a single-sample run, or a `{sample_id: path}` mapping for a cohort.

In [ ]:
tx_path = f'{DATA}/xenium_rcc_core_transcripts.parquet'
txc = transcript_context(adata, tx_path, radius=7.0, sample_col='sample_id', platform='xenium')
print('transcript-context matrix:', txc.shape)
print('mean non-zero genes per cell:', float((txc > 0).sum(1).mean()).__round__(1))

## Concatenate onto counts and train

The model input is `concat(log-norm counts, transcript context)`, so `input_dim` doubles to 732. We log-normalize the counts ourselves and disable normalize / log1p in `TrainConfig` (the context field is already log1p). Because the input is already log-normalized rather than raw counts, we also select the MSE reconstruction (`cell_recon='mse'`, `niche_recon='mse'`, `detection_weight=0`) instead of the count-likelihood default, which expects raw integer counts. We keep `gene_names=()` because the concatenated matrix is not a plain gene panel. A good encoder for transcript context is `mlp_deep` (the library default).

In [ ]:
expr = adata.copy(); sc.pp.normalize_total(expr); sc.pp.log1p(expr)
X_counts = expr.X.toarray() if hasattr(expr.X, 'toarray') else np.asarray(expr.X)
X_in = np.concatenate([X_counts, txc], axis=1).astype(np.float32)
train_ad = ad.AnnData(X=X_in, obs=adata.obs.copy(),
                      obsm={'spatial': np.asarray(adata.obsm['spatial'])})
train_ad.obs_names = adata.obs_names
train_ad.uns['log1p'] = {'base': None}   # mark as already log-normalized

mc = ModelConfig(input_dim=X_in.shape[1], gene_names=(), encoder_type='mlp_deep',
                 cell_recon='mse', niche_recon='mse', detection_weight=0,
                 cell_num_embeddings=256, neighborhood_num_embeddings=32)
# b32k defaults for graph/seed; cell_recon/niche_recon set to 'mse' because the
# transcript-context input is already log-normalized (not raw counts). num_epochs
# (300 -> 20) and batch_size (32768 -> 2048) are shrunk for the demo.
tc = TrainConfig(num_epochs=20, batch_size=2048, k_neighbors=20,
                 spatial_graph='knn_radius', radius=50.0,
                 normalize=False, log1p=False, save_best=False, seed=9)
model, out = nv.train_model(train_ad, 'runs/transcript_context',
                            model_config=mc, train_config=tc, sample_col='sample_id')
print('input_dim (counts + context):', X_in.shape[1])

In [ ]:
cell_idx = out.obs['cell_codebook_idx'].to_numpy()
print('cell codes used:', len(np.unique(cell_idx)), '/ 256')
print('niches used:', out.obs['neighborhood_codebook_idx'].nunique(), '/ 32')
print('embeddings:', out.obsm['X_cell_embedding'].shape)

This single homogeneous TMA core exercises only a few cell codes; codebook fullness grows with the diversity and scale of a real multi-sample cohort (see notebook 01 on MERFISH). The point here is the transcript-context representation and how it is concatenated and trained. To store both, keep `.X` as the 366-gene expression and stash the 732-dim model input in `obsm` for reproducibility.